# 1. Import Libraries and Load the IMDB Dataset

In this step, we import the required libraries and load 3,000 movie reviews from the IMDB dataset.

The sentiment labels are converted into numerical values:

- Positive = 1
- Negative = 0

A fixed random seed is also used to make the experiment reproducible.

In [15]:
!pip install tiktoken -q

import pandas as pd
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import DataLoader, TensorDataset

# Reproducibility
torch.manual_seed(42)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load first 3000 reviews
df = pd.read_csv(
    "IMDB Dataset.csv",
    nrows=3000
)

# Shuffle the reviews
df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Convert sentiment to numerical labels
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print("Original dataset size:", len(df))
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

Using device: cuda
Original dataset size: 3000

Sentiment distribution:
sentiment
positive    1508
negative    1492
Name: count, dtype: int64


# 2. Analyze Review Lengths and Remove Length Outliers

Movie reviews have different numbers of tokens.

We use the tiktoken tokenizer to calculate the length of every review. The longest 5% of reviews are treated as sequence-length outliers and removed.

This reduces extremely long sequences and makes RNN training more efficient.

After removing outliers, the 90th percentile of the cleaned dataset is used to select an appropriate maximum sequence length.

In [16]:
# Load tokenizer
enc = tiktoken.get_encoding("cl100k_base")

# Calculate token length of every review
df["token_length"] = df["review"].apply(
    lambda text: len(enc.encode(text))
)

# Calculate 95th percentile cutoff
outlier_limit = df["token_length"].quantile(0.95)

print("Before cleaning:", len(df))
print("95th percentile cutoff:", outlier_limit)
print("Maximum length before cleaning:", df["token_length"].max())

# Remove longest 5% of reviews
df = df[
    df["token_length"] <= outlier_limit
].copy()

df.reset_index(drop=True, inplace=True)

# Analyze cleaned dataset
lengths = torch.tensor(
    df["token_length"].values,
    dtype=torch.float32
)

p90 = int(torch.quantile(lengths, 0.90).item())
p95 = int(torch.quantile(lengths, 0.95).item())

# Round up MAX_LEN to nearest 50
MAX_LEN = ((p90 + 49) // 50) * 50

print("\nAfter cleaning:", len(df))
print("Reviews removed:", 3000 - len(df))
print("\nCleaned dataset statistics:")
print("Minimum:", lengths.min().item())
print("Average:", lengths.mean().item())
print("Median:", torch.median(lengths).item())
print("90th percentile:", p90)
print("95th percentile:", p95)
print("Maximum:", lengths.max().item())

print("\nSelected MAX_LEN:", MAX_LEN)

Before cleaning: 3000
95th percentile cutoff: 747.1499999999992
Maximum length before cleaning: 1837

After cleaning: 2850
Reviews removed: 150

Cleaned dataset statistics:
Minimum: 22.0
Average: 257.4182434082031
Median: 209.0
90th percentile: 488
95th percentile: 581
Maximum: 747.0

Selected MAX_LEN: 500


# 3. Tokenize Reviews and Create Training and Testing Data

Each review is converted into token IDs using tiktoken.

For reviews longer than MAX_LEN, we preserve information from both parts of the review:

- First half of the tokens
- Last half of the tokens

Shorter reviews are padded with a special padding token.

The cleaned dataset is then split into:

- 80% Training Data
- 20% Testing Data

The split is stratified so that both training and testing datasets contain similar proportions of positive and negative reviews.

In [17]:
PAD_ID = enc.n_vocab

def tokenize(text):
    tokens = enc.encode(text)

    # Keep beginning and ending for long reviews
    if len(tokens) > MAX_LEN:
        half = MAX_LEN // 2
        tokens = tokens[:half] + tokens[-half:]

    # Pad shorter reviews
    tokens += [PAD_ID] * (MAX_LEN - len(tokens))

    return tokens


# Convert reviews to tensors
X = torch.tensor(
    [tokenize(text) for text in df["review"]],
    dtype=torch.long
)

y = torch.tensor(
    df["label"].values,
    dtype=torch.float32
)

# Stratified split without sklearn
torch.manual_seed(42)

positive_idx = torch.where(y == 1)[0]
negative_idx = torch.where(y == 0)[0]

positive_idx = positive_idx[
    torch.randperm(len(positive_idx))
]

negative_idx = negative_idx[
    torch.randperm(len(negative_idx))
]

# 80% from each class for training
pos_split = int(0.8 * len(positive_idx))
neg_split = int(0.8 * len(negative_idx))

train_idx = torch.cat([
    positive_idx[:pos_split],
    negative_idx[:neg_split]
])

test_idx = torch.cat([
    positive_idx[pos_split:],
    negative_idx[neg_split:]
])

# Shuffle train and test indices
train_idx = train_idx[torch.randperm(len(train_idx))]
test_idx = test_idx[torch.randperm(len(test_idx))]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# DataLoaders
train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=32,
    shuffle=False
)

print("MAX_LEN:", MAX_LEN)
print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

print("\nTraining positive ratio:", y_train.mean().item())
print("Testing positive ratio :", y_test.mean().item())

MAX_LEN: 500
Training data: torch.Size([2280, 500])
Testing data : torch.Size([570, 500])

Training positive ratio: 0.5017544031143188
Testing positive ratio : 0.5017544031143188


# 4. Build a Regularized RNN Model

The RNN model contains:

1. An Embedding layer that converts token IDs into dense vectors.
2. A two-layer RNN that processes the sequence.
3. Dropout to reduce overfitting.
4. A Linear layer that produces the final binary classification output.

BCEWithLogitsLoss is used because this is a binary classification problem.

AdamW is used as the optimizer with weight decay for additional regularization.

In [28]:
class RNNModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=enc.n_vocab + 1,
            embedding_dim=128,
            padding_idx=PAD_ID
        )

        self.rnn = nn.RNN(
            input_size=128,
            hidden_size=96,
            num_layers=2,
            batch_first=True,
            dropout=0.4
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(96, 1)

    def forward(self, x):

        x = self.embedding(x)

        _, hidden = self.rnn(x)

        x = self.dropout(hidden[-1])

        return self.fc(x).squeeze(1)


# Create model
model = RNNModel().to(device)

# Binary classification loss
loss_fn = nn.BCEWithLogitsLoss()

# AdamW + L2 regularization
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

print(model)

RNNModel(
  (embedding): Embedding(100278, 128, padding_idx=100277)
  (rnn): RNN(128, 96, num_layers=2, batch_first=True, dropout=0.4)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=96, out_features=1, bias=True)
)


# 5. Train, Evaluate and Apply Early Stopping

The model is trained for a maximum of 20 epochs.

During every epoch, we calculate:

- Training loss
- Training accuracy
- Testing accuracy
- Accuracy gap between training and testing

Gradient clipping is used to stabilize RNN training.

Early stopping stops training when testing accuracy does not improve for several consecutive epochs. The best-performing model is restored at the end.

Finally, the trained model is used to predict the sentiment of a custom movie review.

In [32]:
import copy

EPOCHS = 20
PATIENCE = 4

best_test_acc = 0
best_model_state = None
epochs_without_improvement = 0

for epoch in range(EPOCHS):

    # ================= TRAINING =================
    model.train()

    total_loss = 0
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        # Forward pass
        logits = model(X_batch)

        # Calculate loss
        loss = loss_fn(logits, y_batch)

        # Backpropagation
        loss.backward()

        # Stabilize RNN gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

        predictions = (
            torch.sigmoid(logits) >= 0.5
        ).float()

        train_correct += (
            predictions == y_batch
        ).sum().item()

        train_total += y_batch.size(0)


    # ================= TESTING =================
    model.eval()

    test_correct = 0
    test_total = 0

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)

            predictions = (
                torch.sigmoid(logits) >= 0.5
            ).float()

            test_correct += (
                predictions == y_batch
            ).sum().item()

            test_total += y_batch.size(0)


    # ================= RESULTS =================
    train_acc = 100 * train_correct / train_total
    test_acc = 100 * test_correct / test_total
    gap = train_acc - test_acc

    print(
        f"Epoch {epoch+1:2d}/{EPOCHS} | "
        f"Loss: {total_loss/len(train_loader):.4f} | "
        f"Train: {train_acc:.2f}% | "
        f"Test: {test_acc:.2f}% | "
        f"Gap: {gap:.2f}%"
    )


    # ================= EARLY STOPPING =================
    if test_acc > best_test_acc:

        best_test_acc = test_acc

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print(
                f"\nEarly stopping at epoch {epoch+1}"
            )
            break


# Restore best model
model.load_state_dict(best_model_state)

print(
    f"\nBest Test Accuracy: {best_test_acc:.2f}%"
)


# ================= CUSTOM PREDICTION =================

def predict_sentiment(text):

    model.eval()

    tokens = tokenize(text)

    x = torch.tensor(
        [tokens],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():

        probability = torch.sigmoid(
            model(x)
        ).item()

    sentiment = (
        "Negative"
        if probability <= 0.5
        else "Positive"
    )

    return sentiment, probability


text = "This movie was good."

sentiment, probability = predict_sentiment(text)

print("\nCustom Review:", text)
print("Prediction:", sentiment)
print("Positive Probability:", round(probability, 4))

Epoch  1/20 | Loss: 0.6655 | Train: 55.66% | Test: 51.23% | Gap: 4.43%
Epoch  2/20 | Loss: 0.6769 | Train: 53.55% | Test: 51.23% | Gap: 2.32%
Epoch  3/20 | Loss: 0.6616 | Train: 54.91% | Test: 49.82% | Gap: 5.09%
Epoch  4/20 | Loss: 0.6584 | Train: 53.29% | Test: 51.23% | Gap: 2.06%
Epoch  5/20 | Loss: 0.6506 | Train: 55.00% | Test: 52.46% | Gap: 2.54%
Epoch  6/20 | Loss: 0.6489 | Train: 55.66% | Test: 50.00% | Gap: 5.66%
Epoch  7/20 | Loss: 0.6549 | Train: 53.64% | Test: 50.35% | Gap: 3.29%
Epoch  8/20 | Loss: 0.6463 | Train: 54.69% | Test: 49.82% | Gap: 4.87%
Epoch  9/20 | Loss: 0.6409 | Train: 54.25% | Test: 50.53% | Gap: 3.73%

Early stopping at epoch 9

Best Test Accuracy: 52.46%

Custom Review: This movie was good.
Prediction: Positive
Positive Probability: 0.5671
